In [3]:
import pandas as pd 
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset , DataLoader
import torch.nn as nn
import torch.optim as optim


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

# random seed
torch.manual_seed(42)

# load dataset
df = pd.read_csv('fmnist_small.csv')

# features and labels
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# scaling
X_train = X_train / 255.0
X_test = X_test / 255.0

# custom dataset
class CustomDataset(Dataset):

    def __init__(self, features, labels):

        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):

        return len(self.features)

    def __getitem__(self, index):

        return self.features[index], self.labels[index]

# dataset objects
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

# dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# neural network
class MyNN(nn.Module):

    def __init__(self, num_features):

        super().__init__()

        self.model = nn.Sequential(

            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            

            nn.Linear(64, 10)
        )

    def forward(self, x):

        return self.model(x)

# hyperparameters
epochs = 10
learning_rate = 0.1

# model
model = MyNN(X_train.shape[1])

# loss
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = optim.SGD(
    model.parameters(),
    lr=learning_rate,
    weight_decay=1e-4
)

# training loop
for epoch in range(epochs):

    total_epoch_loss = 0

    for batch_features, batch_labels in train_loader:

        # forward pass
        outputs = model(batch_features)

        # loss
        loss = criterion(outputs, batch_labels)

        # clear old gradients
        optimizer.zero_grad()

        # backpropagation
        loss.backward()

        # update weights
        optimizer.step()

        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(train_loader)

    print(f'Epoch: {epoch+1}, Loss: {avg_loss}')

# evaluation
total = 0
correct = 0

with torch.no_grad():

    for batch_features, batch_labels in test_loader:

        outputs = model(batch_features)

        _, predicted = torch.max(outputs, 1)

        total += batch_labels.shape[0]

        correct += (predicted == batch_labels).sum().item()

accuracy = correct / total

print("Accuracy:", accuracy)

Epoch: 1, Loss: 0.9859935124715169
Epoch: 2, Loss: 0.6983389967679977
Epoch: 3, Loss: 0.6338302338123322
Epoch: 4, Loss: 0.5739411735534667
Epoch: 5, Loss: 0.5367181586225828
Epoch: 6, Loss: 0.5052400282025338
Epoch: 7, Loss: 0.4824169290065765
Epoch: 8, Loss: 0.46870177815357844
Epoch: 9, Loss: 0.4488173471887906
Epoch: 10, Loss: 0.4292281864086787
Accuracy: 0.7866666666666666
